# Task 1

# Loading Data


In [3]:
import pandas as pd
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Define the path to your CSV file
csv_file_path = '/content/drive/MyDrive/Colab Notebooks/fake_reviews_dataset.csv'

# Load the dataset into a pandas DataFrame, trying to handle potential parsing errors
try:
    # Using 'python' engine for better handling of malformed lines and 'on_bad_lines' to skip them
    df = pd.read_csv(csv_file_path, engine='python', on_bad_lines='warn')
    print(f"Successfully loaded the dataset from {csv_file_path}")
    print(f"Dataset shape: {df.shape}")
    print("First 5 rows of the dataset:")
    print(df.head())
except FileNotFoundError:
    print(f"Error: The file at {csv_file_path} was not found. Please ensure the path is correct and Drive is mounted.")
except Exception as e:
    print(f"An error occurred while loading the dataset: {e}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Successfully loaded the dataset from /content/drive/MyDrive/Colab Notebooks/fake_reviews_dataset.csv
Dataset shape: (40432, 4)
First 5 rows of the dataset:
             category  rating label  \
0  Home_and_Kitchen_5     5.0    CG   
1  Home_and_Kitchen_5     5.0    CG   
2  Home_and_Kitchen_5     5.0    CG   
3  Home_and_Kitchen_5     1.0    CG   
4  Home_and_Kitchen_5     5.0    CG   

                                               text_  
0  Love this!  Well made, sturdy, and very comfor...  
1  love it, a great upgrade from the original.  I...  
2  This pillow saved my back. I love the look and...  
3  Missing information on how to use it, but it i...  
4  Very nice set. Good quality. We have had the s...  


# Tokenization using spaCy

In [1]:
!pip install -U spacy
!python -m spacy download en_core_web_sm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 146.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [4]:
import spacy

# Load the spaCy English model
try:
    nlp = spacy.load("en_core_web_sm")
    print("spaCy model 'en_core_web_sm' loaded successfully.")
except Exception as e:
    print(f"Error loading spaCy model: {e}")
    print("Please ensure the model is downloaded by running '!python -m spacy download en_core_web_sm'")
    nlp = None # Set nlp to None if model loading fails

# Define a tokenization function using spaCy
def tokenize_text_spacy(text):
    if nlp is None:
        return [] # Return empty if model failed to load
    if isinstance(text, str):
        doc = nlp(text.lower())
        return [token.text for token in doc if not token.is_space]
    return [] # Return an empty list for non-string values

# Apply tokenization to the 'text_' column using spaCy
df['tokenized_text'] = df['text_'].apply(tokenize_text_spacy)

print("Tokenization complete using spaCy. Here are the first 5 rows with the new 'tokenized_text' column:")
print(df[['text_', 'tokenized_text']].head())

spaCy model 'en_core_web_sm' loaded successfully.
Tokenization complete using spaCy. Here are the first 5 rows with the new 'tokenized_text' column:
                                               text_  \
0  Love this!  Well made, sturdy, and very comfor...   
1  love it, a great upgrade from the original.  I...   
2  This pillow saved my back. I love the look and...   
3  Missing information on how to use it, but it i...   
4  Very nice set. Good quality. We have had the s...   

                                      tokenized_text  
0  [love, this, !, well, made, ,, sturdy, ,, and,...  
1  [love, it, ,, a, great, upgrade, from, the, or...  
2  [this, pillow, saved, my, back, ., i, love, th...  
3  [missing, information, on, how, to, use, it, ,...  
4  [very, nice, set, ., good, quality, ., we, hav...  


# Removing Stop Words

In [5]:
import spacy

# Ensure the spaCy model is loaded (it should be from previous steps)
# If running this cell independently, uncomment and run the following line
# nlp = spacy.load("en_core_web_sm") # Assuming en_core_web_sm is already downloaded

# Define a function to remove stop words
def remove_stopwords(tokens):
    if nlp is None:
        return tokens # Return original tokens if model not loaded
    return [token for token in tokens if token not in nlp.Defaults.stop_words]

# Apply the function to the 'tokenized_text' column
df['filtered_tokens'] = df['tokenized_text'].apply(remove_stopwords)

print("Stop word removal complete. Here are the first 5 rows with 'tokenized_text' and 'filtered_tokens':")
print(df[['tokenized_text', 'filtered_tokens']].head())

Stop word removal complete. Here are the first 5 rows with 'tokenized_text' and 'filtered_tokens':
                                      tokenized_text  \
0  [love, this, !, well, made, ,, sturdy, ,, and,...   
1  [love, it, ,, a, great, upgrade, from, the, or...   
2  [this, pillow, saved, my, back, ., i, love, th...   
3  [missing, information, on, how, to, use, it, ,...   
4  [very, nice, set, ., good, quality, ., we, hav...   

                                     filtered_tokens  
0  [love, !, ,, sturdy, ,, comfortable, ., love, ...  
1  [love, ,, great, upgrade, original, ., couple,...  
2    [pillow, saved, ., love, look, feel, pillow, .]  
3  [missing, information, use, ,, great, product,...  
4      [nice, set, ., good, quality, ., set, months]  


# Vectorization using Bag of words


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Join the filtered tokens back into a single string for TF-IDF Vectorizer
df['text_for_tfidf'] = df['filtered_tokens'].apply(lambda x: ' '.join(x))

# Initialize TfidfVectorizer
tfidf_vectorizer = TfidfVectorizer(max_features=5000) # You can adjust max_features

# Fit and transform the text data
tfidf_matrix = tfidf_vectorizer.fit_transform(df['text_for_tfidf'])

print("TF-IDF vectorization complete.")
print(f"Shape of TF-IDF matrix: {tfidf_matrix.shape}")
print("First 5 TF-IDF vectors (showing non-zero entries for brevity):")

# To display a small part of the sparse matrix
# Convert to dense for viewing, but be careful with large matrices
print(tfidf_matrix[:5].todense())

# Optional: display feature names
# print("Top 10 feature names:")
# print(tfidf_vectorizer.get_feature_names_out()[:10])

TF-IDF vectorization complete.
Shape of TF-IDF matrix: (40432, 5000)
First 5 TF-IDF vectors (showing non-zero entries for brevity):
[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]


# MLP using pytorch

# adding 1 category label

In [7]:
from sklearn.preprocessing import LabelEncoder

# Instantiate LabelEncoder
label_encoder = LabelEncoder()

# Fit and transform the 'category' column
df['category_encoded'] = label_encoder.fit_transform(df['category'])

# Get the number of unique categories
num_classes = len(label_encoder.classes_)

print("Category encoding complete.")
print("First 5 rows with 'category' and 'category_encoded' columns:")
print(df[['category', 'category_encoded']].head())
print(f"Number of unique categories (output classes): {num_classes}")

Category encoding complete.
First 5 rows with 'category' and 'category_encoded' columns:
             category  category_encoded
0  Home_and_Kitchen_5                 3
1  Home_and_Kitchen_5                 3
2  Home_and_Kitchen_5                 3
3  Home_and_Kitchen_5                 3
4  Home_and_Kitchen_5                 3
Number of unique categories (output classes): 10


In [8]:
from sklearn.model_selection import train_test_split
import torch

# Convert TF-IDF matrix and encoded categories to PyTorch tensors
X = torch.tensor(tfidf_matrix.todense(), dtype=torch.float32)
y = torch.tensor(df['category_encoded'].values, dtype=torch.long)

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test: {y_test.shape}")

Shape of X_train: torch.Size([32345, 5000])
Shape of X_test: torch.Size([8087, 5000])
Shape of y_train: torch.Size([32345])
Shape of y_test: torch.Size([8087])


# Differentiation of test and train dataset

In [9]:
from torch.utils.data import Dataset, DataLoader

# Define a custom Dataset class for our data
class TextDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Create Dataset instances
train_dataset = TextDataset(X_train, y_train)
test_dataset = TextDataset(X_test, y_test)

# Define batch size
batch_size = 64 # You can adjust this value

# Create DataLoader instances
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Training dataset size: {len(train_dataset)}")
print(f"Testing dataset size: {len(test_dataset)}")
print(f"Number of batches in training loader: {len(train_loader)}")
print(f"Number of batches in testing loader: {len(test_loader)}")

# Example of one batch
for features, labels in train_loader:
    print(f"Example batch features shape: {features.shape}")
    print(f"Example batch labels shape: {labels.shape}")
    break

Training dataset size: 32345
Testing dataset size: 8087
Number of batches in training loader: 506
Number of batches in testing loader: 127
Example batch features shape: torch.Size([64, 5000])
Example batch labels shape: torch.Size([64])


# Setting architecture


In [14]:
import torch.nn as nn
import torch.nn.functional as F

# Define the MLP model architecture
class MLP(nn.Module):
    def __init__(self, input_size, num_classes):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(input_size, 256)  # First hidden layer
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(256, 128)  # Second hidden layer
        self.fc3 = nn.Linear(128, num_classes) # Output layer (produces logits)
        self.softmax = nn.Softmax(dim=1) # Softmax activation for multiclass probabilities

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.relu(x)
        x = self.fc3(x) # These are raw logits
        x = self.softmax(x) # Apply softmax to get probabilities
        return x

# Get the input size from the TF-IDF matrix (number of features)
input_size = tfidf_matrix.shape[1]

# Instantiate the model
model = MLP(input_size, num_classes)

print(f"MLP Model Architecture:\n{model}")
print(f"Input size: {input_size}")
print(f"Number of output classes: {num_classes}")

MLP Model Architecture:
MLP(
  (fc1): Linear(in_features=5000, out_features=256, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=256, out_features=128, bias=True)
  (fc3): Linear(in_features=128, out_features=10, bias=True)
  (softmax): Softmax(dim=1)
)
Input size: 5000
Number of output classes: 10


In [15]:
import torch.optim as optim

# Define the loss function (CrossEntropyLoss for multi-class classification)
criterion = nn.CrossEntropyLoss()

# Define the optimizer (Adam is a good general-purpose optimizer)
optimizer = optim.Adam(model.parameters(), lr=0.001) # You can adjust the learning rate (lr)

print("Loss function and optimizer defined.")
print(f"Criterion: {criterion}")
print(f"Optimizer: {optimizer.__class__.__name__}")

Loss function and optimizer defined.
Criterion: CrossEntropyLoss()
Optimizer: Adam


# Training


In [16]:
num_epochs = 50 # You can adjust the number of epochs

# Move model to appropriate device (CPU or GPU if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print(f"Training model on: {device}")

# Training loop
for epoch in range(num_epochs):
    model.train() # Set the model to training mode
    running_loss = 0.0
    correct_train = 0
    total_train = 0

    for i, (inputs, labels) in enumerate(train_loader):
        inputs, labels = inputs.to(device), labels.to(device)

        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        # Backward pass and optimize
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()

    train_accuracy = 100 * correct_train / total_train
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {running_loss/len(train_loader):.4f}, Train Accuracy: {train_accuracy:.2f}%")

    # Evaluation loop (on test set)
    model.eval() # Set the model to evaluation mode
    correct_test = 0
    total_test = 0
    with torch.no_grad(): # Disable gradient calculation for evaluation
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total_test += labels.size(0)
            correct_test += (predicted == labels).sum().item()

    test_accuracy = 100 * correct_test / total_test
    print(f"Test Accuracy after Epoch {epoch+1}: {test_accuracy:.2f}%")

print("Finished Training and Evaluation")

Training model on: cuda
Epoch 1/50, Loss: 1.8827, Train Accuracy: 61.34%
Test Accuracy after Epoch 1: 75.32%
Epoch 2/50, Loss: 1.6638, Train Accuracy: 80.57%
Test Accuracy after Epoch 2: 75.99%
Epoch 3/50, Loss: 1.6154, Train Accuracy: 85.22%
Test Accuracy after Epoch 3: 76.13%
Epoch 4/50, Loss: 1.5930, Train Accuracy: 87.26%
Test Accuracy after Epoch 4: 76.18%
Epoch 5/50, Loss: 1.5781, Train Accuracy: 88.67%
Test Accuracy after Epoch 5: 75.65%
Epoch 6/50, Loss: 1.5692, Train Accuracy: 89.49%
Test Accuracy after Epoch 6: 75.54%
Epoch 7/50, Loss: 1.5633, Train Accuracy: 90.03%
Test Accuracy after Epoch 7: 75.40%
Epoch 8/50, Loss: 1.5591, Train Accuracy: 90.38%
Test Accuracy after Epoch 8: 75.05%
Epoch 9/50, Loss: 1.5560, Train Accuracy: 90.67%
Test Accuracy after Epoch 9: 75.15%
Epoch 10/50, Loss: 1.5530, Train Accuracy: 90.94%
Test Accuracy after Epoch 10: 75.10%
Epoch 11/50, Loss: 1.5507, Train Accuracy: 91.16%
Test Accuracy after Epoch 11: 74.90%
Epoch 12/50, Loss: 1.5487, Train Accu

# Evaluation for Task 1 Results



In [18]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
import numpy as np

# Set the model to evaluation mode
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Convert lists to numpy arrays
all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

# Calculate overall accuracy
overall_accuracy = accuracy_score(all_labels, all_preds)
print(f"Overall Accuracy: {overall_accuracy:.4f}")

# Get per-class precision, recall, f1-score
# 'num_classes' was defined earlier during LabelEncoder. We need the original class names
# If label_encoder is available, use its classes; otherwise, define range
if 'label_encoder' in globals():
    target_names = label_encoder.classes_
else:
    target_names = [f'Class {i}' for i in range(num_classes)]

print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=target_names))

# Optional: Calculate macro/weighted average metrics if needed
macro_precision = precision_score(all_labels, all_preds, average='macro')
macro_recall = recall_score(all_labels, all_preds, average='macro')
macro_f1 = f1_score(all_labels, all_preds, average='macro')
print(f"Macro Precision: {macro_precision:.4f}")
print(f"Macro Recall: {macro_recall:.4f}")
print(f"Macro F1-Score: {macro_f1:.4f}")

Overall Accuracy: 0.7421

Classification Report:
                              precision    recall  f1-score   support

                     Books_5       0.71      0.67      0.69       874
Clothing_Shoes_and_Jewelry_5       0.73      0.82      0.77       770
               Electronics_5       0.80      0.79      0.79       798
          Home_and_Kitchen_5       0.64      0.70      0.67       811
              Kindle_Store_5       0.74      0.76      0.75       946
             Movies_and_TV_5       0.88      0.84      0.86       717
              Pet_Supplies_5       0.86      0.84      0.85       851
       Sports_and_Outdoors_5       0.60      0.58      0.59       789
Tools_and_Home_Improvement_5       0.71      0.63      0.67       772
            Toys_and_Games_5       0.77      0.80      0.79       759

                    accuracy                           0.74      8087
                   macro avg       0.74      0.74      0.74      8087
                weighted avg       0.74

#


#

#

# **Task 2**

In [19]:
from sklearn.preprocessing import LabelEncoder

# Instantiate LabelEncoder
label_encoder_binary = LabelEncoder()

# Fit and transform the 'label' column
df['binary_label'] = label_encoder_binary.fit_transform(df['label'])

print("Binary encoding of 'label' column complete.")
print("First 5 rows with original 'label' and new 'binary_label' column:")
print(df[['label', 'binary_label']].head())

# Optional: Display the mapping
print("\nLabel mapping:")
for i, label in enumerate(label_encoder_binary.classes_):
    print(f"{label}: {i}")

Binary encoding of 'label' column complete.
First 5 rows with original 'label' and new 'binary_label' column:
  label  binary_label
0    CG             0
1    CG             0
2    CG             0
3    CG             0
4    CG             0

Label mapping:
CG: 0
OR: 1


In [20]:
from sklearn.model_selection import train_test_split
import torch

# Convert TF-IDF matrix to a PyTorch tensor
X_binary = torch.tensor(tfidf_matrix.todense(), dtype=torch.float32)

# Convert binary_label column to a PyTorch tensor
y_binary = torch.tensor(df['binary_label'].values, dtype=torch.long)

# Split the data into training and testing sets for binary classification
X_train_binary, X_test_binary, y_train_binary, y_test_binary = train_test_split(
    X_binary, y_binary, test_size=0.2, random_state=42, stratify=y_binary
)

print(f"Shape of X_train_binary: {X_train_binary.shape}")
print(f"Shape of X_test_binary: {X_test_binary.shape}")
print(f"Shape of y_train_binary: {y_train_binary.shape}")
print(f"Shape of y_test_binary: {y_test_binary.shape}")

Shape of X_train_binary: torch.Size([32345, 5000])
Shape of X_test_binary: torch.Size([8087, 5000])
Shape of y_train_binary: torch.Size([32345])
Shape of y_test_binary: torch.Size([8087])


In [21]:
from torch.utils.data import Dataset, DataLoader

# Create Dataset instances for binary classification
train_dataset_binary = TextDataset(X_train_binary, y_train_binary)
test_dataset_binary = TextDataset(X_test_binary, y_test_binary)

# Define batch size (already defined as batch_size = 64 from previous steps)

# Create DataLoader instances for binary classification
train_loader_binary = DataLoader(train_dataset_binary, batch_size=batch_size, shuffle=True)
test_loader_binary = DataLoader(test_dataset_binary, batch_size=batch_size, shuffle=False)

print(f"Training dataset size for binary task: {len(train_dataset_binary)}")
print(f"Testing dataset size for binary task: {len(test_dataset_binary)}")
print(f"Number of batches in binary training loader: {len(train_loader_binary)}")
print(f"Number of batches in binary testing loader: {len(test_loader_binary)}")

# Example of one batch from the binary training loader
for features_binary, labels_binary in train_loader_binary:
    print(f"Example binary batch features shape: {features_binary.shape}")
    print(f"Example binary batch labels shape: {labels_binary.shape}")
    break

Training dataset size for binary task: 32345
Testing dataset size for binary task: 8087
Number of batches in binary training loader: 506
Number of batches in binary testing loader: 127
Example binary batch features shape: torch.Size([64, 5000])
Example binary batch labels shape: torch.Size([64])


In [45]:
import torch.nn as nn
import torch.nn.functional as F

# Define the MLP model architecture for binary classification
class BinaryMLP(nn.Module):
    def __init__(self, input_size, num_classes=2):
        super(BinaryMLP, self).__init__()
        self.fc1 = nn.Linear(input_size, 256)  # First hidden layer
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(256, 128)  # Second hidden layer
        self.fc3 = nn.Linear(128, num_classes) # Output layer with 2 classes for binary classification

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.relu(x)
        x = self.fc3(x) # These are raw logits
        return x

# Get the input size from the TF-IDF matrix (number of features)
input_size_binary = tfidf_matrix.shape[1]

# Instantiate the binary classification model
binary_model = BinaryMLP(input_size_binary, num_classes=2)

print(f"Binary MLP Model Architecture:\n{binary_model}")
print(f"Input size: {input_size_binary}")
print(f"Number of output classes for binary classification: 2")

Binary MLP Model Architecture:
BinaryMLP(
  (fc1): Linear(in_features=5000, out_features=256, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=256, out_features=128, bias=True)
  (fc3): Linear(in_features=128, out_features=2, bias=True)
)
Input size: 5000
Number of output classes for binary classification: 2


In [46]:
import torch.optim as optim

# Define the loss function for binary classification (CrossEntropyLoss is suitable for 2 classes as well)
criterion_binary = nn.CrossEntropyLoss()

# Define the optimizer for the binary model
optimizer_binary = optim.Adam(binary_model.parameters(), lr=0.001) # You can adjust the learning rate (lr)

print("Loss function and optimizer defined for binary classification.")
print(f"Binary Criterion: {criterion_binary}")
print(f"Binary Optimizer: {optimizer_binary.__class__.__name__}")

Loss function and optimizer defined for binary classification.
Binary Criterion: CrossEntropyLoss()
Binary Optimizer: Adam


In [48]:
num_epochs_binary = 10 # You can adjust the number of epochs for binary classification, reducing for debugging

# Move model to appropriate device (CPU or GPU if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
binary_model.to(device)

print(f"Training binary model on: {device}")

# Training loop for binary classification
for epoch in range(num_epochs_binary):
    binary_model.train() # Set the model to training mode
    running_loss_binary = 0.0
    correct_train_binary = 0
    total_train_binary = 0

    for i, (inputs, labels) in enumerate(train_loader_binary):
        inputs, labels = inputs.to(device), labels.to(device)

        # Zero the parameter gradients
        optimizer_binary.zero_grad()

        # Forward pass
        outputs = binary_model(inputs)
        loss = criterion_binary(outputs, labels)

        # Backward pass
        loss.backward()

        # Gradient clipping to prevent exploding gradients
        torch.nn.utils.clip_grad_norm_(binary_model.parameters(), max_norm=1.0) # You can adjust max_norm

        # Optimize
        optimizer_binary.step()

        running_loss_binary += loss.item()

        _, predicted = torch.max(outputs.data, 1)
        total_train_binary += labels.size(0)
        correct_train_binary += (predicted == labels).sum().item()

    train_accuracy_binary = 100 * correct_train_binary / total_train_binary
    print(f"Epoch {epoch+1}/{num_epochs_binary}, Binary Loss: {running_loss_binary/len(train_loader_binary):.4f}, Binary Train Accuracy: {train_accuracy_binary:.2f}%")

    # Evaluation loop (on test set)
    binary_model.eval() # Set the model to evaluation mode
    correct_test_binary = 0
    total_test_binary = 0
    with torch.no_grad(): # Disable gradient calculation for evaluation
        for inputs, labels in test_loader_binary:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = binary_model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total_test_binary += labels.size(0)
            correct_test_binary += (predicted == labels).sum().item()

    test_accuracy_binary = 100 * correct_test_binary / total_test_binary
    print(f"Binary Test Accuracy after Epoch {epoch+1}: {test_accuracy_binary:.2f}%")

print("Finished Binary Model Training and Evaluation")

AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [25]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
import numpy as np

# Set the binary model to evaluation mode
binary_model.eval()

all_preds_binary = []
all_labels_binary = []

with torch.no_grad():
    for inputs, labels in test_loader_binary:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = binary_model(inputs)
        _, predicted = torch.max(outputs.data, 1)

        all_preds_binary.extend(predicted.cpu().numpy())
        all_labels_binary.extend(labels.cpu().numpy())

# Convert lists to numpy arrays
all_preds_binary = np.array(all_preds_binary)
all_labels_binary = np.array(all_labels_binary)

# Calculate overall accuracy
overall_accuracy_binary = accuracy_score(all_labels_binary, all_preds_binary)
print(f"Overall Binary Accuracy: {overall_accuracy_binary:.4f}")

# Get per-class precision, recall, f1-score
# Use label_encoder_binary to get target names for the binary classification
if 'label_encoder_binary' in globals():
    target_names_binary = label_encoder_binary.classes_
else:
    target_names_binary = [f'Class {i}' for i in range(2)] # Default to Class 0, Class 1

print("\nBinary Classification Report:")
print(classification_report(all_labels_binary, all_preds_binary, target_names=target_names_binary))

# Optional: Calculate macro/weighted average metrics if needed
macro_precision_binary = precision_score(all_labels_binary, all_preds_binary, average='macro')
macro_recall_binary = recall_score(all_labels_binary, all_preds_binary, average='macro')
macro_f1_binary = f1_score(all_labels_binary, all_preds_binary, average='macro')
print(f"Macro Binary Precision: {macro_precision_binary:.4f}")
print(f"Macro Binary Recall: {macro_recall_binary:.4f}")
print(f"Macro Binary F1-Score: {macro_f1_binary:.4f}")

Overall Binary Accuracy: 0.8683

Binary Classification Report:
              precision    recall  f1-score   support

          CG       0.87      0.87      0.87      4044
          OR       0.87      0.87      0.87      4043

    accuracy                           0.87      8087
   macro avg       0.87      0.87      0.87      8087
weighted avg       0.87      0.87      0.87      8087

Macro Binary Precision: 0.8683
Macro Binary Recall: 0.8683
Macro Binary F1-Score: 0.8683
